[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/04_categorization.ipynb)

# R2-Q2 Week 4 — Apply the taxonomy and report the failure-mode distribution

This notebook applies the four numeric predicates committed in Week 1 to the Grad-CAM heatmaps produced in Week 3, assigning each misclassification to one of five categories — symptom-attended-but-wrong-class, background-attended, lighting-attended, leaf-shape-attended, or other/ambiguous — following the first-match-wins scheme with multi-fires routing to "other." The output is the failure-mode distribution across the misclassification set, reported at the overall taxonomy level rather than per-class, and the methods-section text documenting how each predicate was operationalized.

By the end of this notebook you will have:

- A `categorizations.parquet` containing one row per misclassification with columns recording which predicates fired, the assigned category, and the derived quantities (m_out, m_perimeter, lighting_correlation, m_in, concentration) that drove the assignment — so a reader can audit any individual categorization.
- A `taxonomy_distribution.json` recording the count and fraction of misclassifications in each of the five categories, the conjunction-handling statistics (how many images fired multiple predicates and routed to "other"), and the headline interpretation sentence describing what the failure-mode distribution looks like.
- A `taxonomy_distribution.png` bar chart showing the five-category distribution with the "other" bucket visually distinguished — so a reader can see at a glance whether failures clustered into recognizable patterns or whether the taxonomy itself needs revision.

## Before you run anything: switch to a GPU runtime

This notebook uses a large vision model (the Segment Anything Model,
or SAM) to separate the leaf from the background in each of the 81
misclassified images. SAM runs much faster on a GPU than on a CPU,
and 81 images on CPU adds up to a long wait. Colab's default runtime
is CPU-only, so you'll need to switch.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive and set up irilab2026 with a GPU.
import irilab2026 as iri
iri.setup(gpu_required=True)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 1 — Load the inputs from the earlier notebooks

This notebook reads three files produced by the earlier notebooks in
this series:

- `precommit.json`, from `01_pre_commits.ipynb`
- `working_set.parquet`, from `02_load_and_filter.ipynb`
- `gradcam_metadata.parquet`, from `03_sanity_checks_and_gradcam.ipynb`

The three subsections below load each in turn, with the validation
needed to catch the most common ways they can be missing or wrong.
All three loads share the same posture: try to read the file, raise
a helpful error if it's missing, validate just enough to catch the
obvious failures, and print what got loaded so you can see the
structure without inspecting it manually.

Deeper validation of the precommit's internal structure happens in
Sections 3, 4, and 5 — at the point where each sub-field is actually
used. That keeps error messages close to the symptoms that produce
them.

### 1.1) Load and validate the pre-commit

The precommit is the contract from Week 1 — the document that captures
every decision about how this categorization will run. N04 reads from
it in every later section: the leaf-segmentation method (Section 3),
the five derived quantities (Section 4), the four predicates with
their thresholds and order (Section 5), and the aggregation level for
the final summary (Section 6).

The validation below is intentionally light. It confirms the four
top-level fields the precommit promises (`taxonomy`,
`categorization_procedure`, `aggregation_level`, `sanity_checks`) —
enough to say "this is a real precommit, not an empty file" — plus
one slightly deeper check that the taxonomy's category list is
populated, since every later section iterates over it. Validation
of specific sub-fields (the thresholds, the predicate order, the
segmentation parameters) happens in the section that reads each one.

In [ ]:
### 1.1) Load and validate the pre-commit

import json

precommit_path = OUTPUT_DIR / "precommit.json"

try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {precommit_path}. "
        "This file is produced by 01_pre_commits.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light validation: the four locked top-level keys must all be present.
# Each later section validates the specific sub-fields it uses.
expected_keys = {
    "taxonomy",
    "categorization_procedure",
    "sanity_checks",
    "aggregation_level",
}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing top-level keys: {sorted(missing)}. "
        "This usually means N01 didn't complete — "
        "re-run 01_pre_commits.ipynb to regenerate the file."
    )

# Slightly deeper: the taxonomy categories list is what every later
# section iterates over. Check here that it's present and non-empty
# so the print below doesn't crash with a noisy KeyError.
if not precommit["taxonomy"].get("categories"):
    raise KeyError(
        "precommit['taxonomy']['categories'] is missing or empty. "
        "N01 Section 4 didn't complete — re-run 01_pre_commits.ipynb."
    )

print(f"Loaded: {precommit_path}")
print(f"  top-level keys: {sorted(precommit.keys())}")
print(f"  taxonomy ({len(precommit['taxonomy']['categories'])} categories):")
for cat in precommit["taxonomy"]["categories"]:
    print(f"    - {cat}")

### 1.2) Load the working set

N02 wrote `working_set.parquet` to this question's output directory:
the rows of the PlantDoc prediction table where the model
misclassified at the category level. N03 read this same file to
generate the Grad-CAM heatmaps; N04 reads it again to pair each
misclassification with its heatmap and its source image.

This notebook uses one column from the working set: `filename`. It
serves two purposes — joining each row to its heatmap in the next
subsection, and looking up the source image from the PlantDoc HF
Dataset in Section 3 (the same approach N03 used to feed images
into Grad-CAM). All the other columns (`true_category`, `host`,
`disease`, etc.) pass through to the per-image output table N04
produces in Section 6, so a reader can audit any individual
categorization against the original prediction data.

In [ ]:
### 1.2) Load the working set

import pandas as pd

working_set_path = OUTPUT_DIR / "working_set.parquet"

try:
    working_set = pd.read_parquet(working_set_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {working_set_path}. "
        "This file is produced by 02_load_and_filter.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light schema validation: the one column N04 strictly depends on.
# `filename` is the join key for the heatmap metadata and also the
# lookup key into the PlantDoc HF Dataset that Sections 3 and 4 use
# to pull source images.
required_columns = {"filename"}
missing = required_columns - set(working_set.columns)
if missing:
    raise KeyError(
        f"working_set.parquet is missing required columns: {sorted(missing)}. "
        "This usually means N02 wrote an older or partial schema — "
        "re-run 02_load_and_filter.ipynb to regenerate the file."
    )

# An empty working set means there are no misclassifications to
# categorize — methodologically interesting but operationally a stop
# condition for N04.
if len(working_set) == 0:
    raise ValueError(
        "working_set.parquet has zero rows. "
        "There are no misclassifications to categorize — "
        "if this is unexpected, check N02's filter logic."
    )

print(f"Loaded: {working_set_path}")
print(f"  rows: {len(working_set)}")
print(f"  columns: {sorted(working_set.columns)}")

### 1.3) Load the Grad-CAM metadata

N03 wrote one `.npy` file per misclassification — 81 heatmaps at 7×7
resolution, stored under `OUTPUT_DIR / "heatmaps"`. Alongside them
N03 wrote `gradcam_metadata.parquet`, a table that joins each heatmap
file back to its working-set row. Section 4 will load each heatmap
as it processes the corresponding image; this section confirms the
metadata is well-formed and the files are where the metadata says
they are.

The validation here is more aggressive than the previous two loads
because the file-existence check is genuinely cheap — 81 stat calls
take well under a second — and catching a missing heatmap here
saves the better part of a minute of SAM segmentation in Section 3
before the error would otherwise surface in Section 4.

Three checks, in order:

1. Every required column is present.
2. Every working-set row has a matching entry in the metadata.
3. Every referenced `.npy` file actually exists on disk.

In [ ]:
### 1.3) Load the Grad-CAM metadata

from pathlib import Path

gradcam_metadata_path = OUTPUT_DIR / "gradcam_metadata.parquet"

try:
    gradcam_metadata = pd.read_parquet(gradcam_metadata_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {gradcam_metadata_path}. "
        "This file is produced by 03_sanity_checks_and_gradcam.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

if len(gradcam_metadata) == 0:
    raise ValueError(
        f"{gradcam_metadata_path} has zero rows. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the file."
    )

# Check 1: required columns.
required_columns = {"filename", "heatmap_path"}
missing = required_columns - set(gradcam_metadata.columns)
if missing:
    raise KeyError(
        f"gradcam_metadata.parquet is missing required columns: "
        f"{sorted(missing)}. This usually means N03 wrote an older or "
        "partial schema — re-run 03_sanity_checks_and_gradcam.ipynb "
        "to regenerate the file."
    )

# Check 2: every working-set row has a matching heatmap entry.
working_filenames = set(working_set["filename"])
heatmap_filenames = set(gradcam_metadata["filename"])
missing_entries = working_filenames - heatmap_filenames
if missing_entries:
    raise KeyError(
        f"{len(missing_entries)} working-set row(s) have no entry in "
        f"gradcam_metadata. First few: {sorted(missing_entries)[:3]}. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the "
        "metadata."
    )

# Check 3: every referenced .npy file exists on disk. Paths may be
# stored as absolute or relative; resolve relative paths against
# OUTPUT_DIR before checking.
missing_files = []
for p in gradcam_metadata["heatmap_path"]:
    resolved = Path(p)
    if not resolved.is_absolute():
        resolved = OUTPUT_DIR / resolved
    if not resolved.exists():
        missing_files.append(str(resolved))

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} heatmap file(s) referenced in "
        f"gradcam_metadata are missing from disk. "
        f"First few: {missing_files[:3]}. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the "
        "heatmaps."
    )

print(f"Loaded: {gradcam_metadata_path}")
print(f"  rows: {len(gradcam_metadata)}")
print(f"  all {len(working_set)} working-set rows have matching heatmaps — OK")
print(f"  all {len(gradcam_metadata)} heatmap files present on disk — OK")

## 2 — Segment the leaf in each image

Sections 3 and 4 need to know which pixels in each image are leaf and
which are not. The five derived quantities all reference the leaf
region — `m_out` is the fraction of attention mass that falls outside
the leaf, `m_perimeter` is the fraction in a band near the leaf's edge,
and so on. Without a mask there's no in-or-out distinction and no
derived quantity to compare against the precommit's thresholds.

This section produces that mask for every image in the working set.
The precommit names SAM (the Segment Anything Model) as the
segmentation method, with the prompting strategy "single point at
image center" and a fallback to color-based thresholding if SAM
proves unreliable. Reliability is checked against five hand-segmented
reference masks shipped with the program.

The five subsections:

| Sub | What it does |
|---|---|
| 2.1 | Set up SAM — install, download the checkpoint, build the predictor, define `segment_leaf` |
| 2.2 | Run `segment_leaf` on all 81 working-set images and save raw SAM masks |
| 2.3 | Validate against five hand-segmented reference masks — compute IoU per image |
| 2.4 | Apply the pass-or-fallback decision per the precommit's criterion |
| 2.5 | Save the final per-image masks as PNGs for Section 3 |

By the end of this section, `OUTPUT_DIR / "leaf_masks"` will contain 81
PNG files, one per working-set image, plus a `segmentation_summary.json`
recording the IoU statistics, the pass/fail verdict, and which images
(if any) ended up using the color fallback rather than SAM.

### 2.1) Set up SAM

This subsection assembles the machinery the rest of Section 3 will
use: the SAM model loaded on GPU, a PlantDoc image loader (the same
shape as N03's, but returning the raw PIL image rather than a
normalized tensor), and a `segment_leaf` helper that wraps SAM's
predict API to match the precommit's prompting strategy.

A few things worth knowing upfront:

**Why `vit_h`.** SAM ships in three sizes (`vit_b`, `vit_l`, `vit_h`).
`vit_h` is the largest and most accurate, especially on cluttered
field-condition backgrounds — which is exactly what PlantDoc images
have. The cost is a one-time 2.4 GB download per Drive (cached
afterward) and a few hundred milliseconds of inference per image on
a T4 GPU.

**The checkpoint is cached in your Drive.** First call downloads the
file into `OUTPUT_DIR`; subsequent runtimes pick it up from there.
Restarting the Colab runtime doesn't trigger another download.

**`segment_leaf` returns a boolean mask.** Same height and width as
the input image, `True` where SAM identifies the leaf. The prompting
strategy is the one committed in N01: a single foreground point at
the image center, with SAM's three candidate masks scored internally
and the highest-scoring one returned.

The smoke test at the end runs `segment_leaf` on the first working-set
image and plots the result, so you can see SAM is working before
committing to the full 81-image run in 3.2.

In [ ]:
### 2.1) Install segment-anything

# segment-anything is not a dependency of irilab2026 (it's only used
# in this notebook), so we pip-install it here. On a fresh runtime
# this takes ~10 seconds; on a warm runtime pip recognizes it's
# already installed and exits immediately.

!pip install -q segment-anything

In [ ]:
### 2.2) Load PlantDoc and define the image loader

# Load PlantDoc metadata and image data. First call on a fresh runtime
# downloads ~950 MB from Hugging Face and caches to Drive; subsequent
# calls hit the cache. The metadata DataFrame has a `filename` column
# that matches the working set's join key.
pd_metadata, pd_hf_dataset = iri.load_plantdoc()
print(f"PlantDoc loaded: {len(pd_metadata)} rows")

# Build a filename → pandas-index lookup so we can pull a specific
# image by its filename in O(1). The HF dataset is index-aligned to
# the metadata, so the pandas index doubles as the hf_dataset position.
filename_to_pd_index = dict(zip(pd_metadata["filename"], pd_metadata.index))


def load_pd_image(filename):
    """Load a PlantDoc image by filename, returning the raw PIL image
    at the original resolution.

    N04 needs the raw image rather than N03's eval-transformed tensor
    because SAM and the brightness analysis in Section 4 both work on
    the full-resolution RGB pixels, not 224×224 normalized tensors.
    """
    if filename not in filename_to_pd_index:
        raise KeyError(
            f"{filename} not found in PlantDoc metadata. "
            "Check that the working set was built against the same "
            "PD dataset version."
        )
    pd_idx = filename_to_pd_index[filename]
    image_pil = pd_hf_dataset[int(pd_idx)]["image"]
    return image_pil.convert("RGB")


# Quick loader smoke test — confirm it returns a PIL image of plausible size.
_first = working_set["filename"].iloc[0]
_test = load_pd_image(_first)
print(f"Loader smoke test: {_first} → PIL image, size {_test.size}, mode {_test.mode}")

In [ ]:
### 2.3) Download the SAM checkpoint and build the predictor

import torch
from segment_anything import sam_model_registry, SamPredictor

SAM_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
SAM_CHECKPOINT_PATH = OUTPUT_DIR / "sam_vit_h_4b8939.pth"

if not SAM_CHECKPOINT_PATH.exists():
    print(f"Downloading SAM vit_h checkpoint to {SAM_CHECKPOINT_PATH}")
    print("This is a one-time download (~2.4 GB). Subsequent runtimes hit the cache.")
    torch.hub.download_url_to_file(
        SAM_CHECKPOINT_URL,
        str(SAM_CHECKPOINT_PATH),
        progress=True,
    )
    print("Download complete.")
else:
    print(f"Using cached SAM checkpoint at {SAM_CHECKPOINT_PATH}")

# Build the model and wrap it in a predictor. The .to("cuda") move
# puts the model on GPU so the per-image inference in 3.2 runs in a
# few hundred milliseconds rather than tens of seconds.
sam = sam_model_registry["vit_h"](checkpoint=str(SAM_CHECKPOINT_PATH))
sam.to("cuda")
predictor = SamPredictor(sam)

print(f"SAM vit_h loaded on {next(sam.parameters()).device}")

In [ ]:
### 2.4) Define segment_leaf

import numpy as np


def segment_leaf(image_pil):
    """Run SAM on a PIL image and return the predicted leaf mask.

    Uses the precommit's prompting strategy: a single foreground point
    at the image center. SAM returns three candidate masks at different
    nested scales; this function takes the one SAM scores highest.

    Parameters
    ----------
    image_pil : PIL.Image
        The source image. RGB conversion is applied defensively.

    Returns
    -------
    mask : numpy.ndarray
        Boolean mask, shape (H, W), True where SAM identifies the leaf.
    """
    image_pil = image_pil.convert("RGB")
    image_np = np.array(image_pil)
    H, W = image_np.shape[:2]

    # set_image preprocesses the image (resize, normalize) and runs
    # the SAM image encoder. This is the expensive step (~100 ms on
    # T4); everything after is cheap.
    predictor.set_image(image_np)

    # Single foreground point at image center.
    center_point = np.array([[W // 2, H // 2]])
    foreground_label = np.array([1])  # 1 = foreground, 0 = background

    # multimask_output=True returns three candidate masks at different
    # nested scales (small, medium, large) along with SAM's own quality
    # score for each. The highest-scoring one is the model's most
    # confident leaf boundary for this prompt.
    masks, scores, _ = predictor.predict(
        point_coords=center_point,
        point_labels=foreground_label,
        multimask_output=True,
    )

    best_idx = int(np.argmax(scores))
    return masks[best_idx]

In [ ]:
### 2.5) Smoke-test segment_leaf on the first working-set image

import matplotlib.pyplot as plt

test_filename = working_set["filename"].iloc[0]
test_image = load_pd_image(test_filename)
test_mask = segment_leaf(test_image)

# Verify the mask shape and dtype match expectations before plotting.
test_image_np = np.array(test_image)
assert test_mask.shape == test_image_np.shape[:2], (
    f"Mask shape {test_mask.shape} doesn't match image shape "
    f"{test_image_np.shape[:2]}"
)
assert test_mask.dtype == bool, f"Expected boolean mask, got {test_mask.dtype}"

# Show the image and the mask side by side.
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(test_image_np)
axes[0].set_title(f"Image: {test_filename}")
axes[0].axis("off")
axes[1].imshow(test_mask, cmap="gray")
axes[1].set_title(f"SAM mask ({test_mask.mean():.1%} of pixels)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(
    f"Smoke test passed — mask shape {test_mask.shape}, "
    f"{test_mask.mean():.1%} foreground"
)

## 3 — Run SAM on the full working set

This subsection runs `segment_leaf` on each of the 81 working-set
images and saves the result as a PNG. The per-image saves happen
inside the loop (not batched at the end), so a kernel crash partway
through doesn't lose progress — masks already on disk stay there.

What gets saved:

- 81 PNGs in `OUTPUT_DIR / "leaf_masks"`, one per working-set image,
  named `<filename>.png` to match the working set's `filename`
  column. Mode is grayscale (0 = not leaf, 255 = leaf), so any image
  viewer can open them for visual spot-checking.
- A `leaf_masks_metadata.parquet` recording the file path and
  foreground fraction (the share of pixels SAM marked as leaf) for
  each image. Subsections 2.3 and 2.4 add columns to this table.

Total runtime is typically around a minute on T4 — SAM inference is
a few hundred milliseconds per image, with the rest spent on file
I/O to Drive.

In [ ]:
### 3.1) Segment every working-set image

from tqdm.auto import tqdm
from PIL import Image
import time

mask_dir = OUTPUT_DIR / "leaf_masks"
mask_dir.mkdir(exist_ok=True)

# Track per-image stats so we can persist them in the metadata parquet
# and surface a distribution plot in the next cell.
per_image_records = []

start_time = time.time()
for _, row in tqdm(working_set.iterrows(), total=len(working_set), desc="Segmenting"):
    filename = row["filename"]
    image_pil = load_pd_image(filename)
    mask = segment_leaf(image_pil)

    # Save as grayscale PNG (0 = not leaf, 255 = leaf). Mode 'L' is
    # universally readable by any image viewer; after PNG compression
    # the file size is essentially the same as mode '1' (binary).
    mask_path = mask_dir / f"{filename}.png"
    Image.fromarray((mask * 255).astype(np.uint8), mode="L").save(mask_path)

    per_image_records.append({
        "filename": filename,
        "mask_path": str(mask_path),
        "fg_fraction": float(mask.mean()),
    })

elapsed = time.time() - start_time

# Persist the per-image metadata. 2.3 and 2.4 will reread and extend
# this table; Section 3 will reference mask_path to load each mask.
leaf_masks_metadata = pd.DataFrame(per_image_records)
metadata_path = OUTPUT_DIR / "leaf_masks_metadata.parquet"
leaf_masks_metadata.to_parquet(metadata_path)

print(f"\nSegmented {len(working_set)} images in {elapsed:.1f}s")
print(f"  ({elapsed / len(working_set):.2f}s per image average)")
print(f"  Masks saved to: {mask_dir}")
print(f"  Metadata saved to: {metadata_path}")

In [ ]:
### 3.2) Plot the distribution of foreground fractions

# A quick visual of how much of each image SAM marked as leaf. We
# expect a unimodal distribution centered somewhere in the 20–60%
# range with a tail toward smaller values (images where the leaf is
# a smaller subject against a lot of background). A bimodal or
# heavily skewed distribution would suggest SAM struggling on a
# subset of images.

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    leaf_masks_metadata["fg_fraction"],
    bins=20,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_xlabel("Foreground fraction (share of pixels SAM marked as leaf)")
ax.set_ylabel("Number of images")
ax.set_title(
    "Distribution of SAM mask foreground fractions across "
    f"the {len(leaf_masks_metadata)} working-set images"
)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

# Summary stats.
fg_summary = leaf_masks_metadata["fg_fraction"].describe()
print("Foreground fraction summary:")
print(f"  median:  {fg_summary['50%']:.1%}")
print(f"  IQR:     {fg_summary['25%']:.1%} – {fg_summary['75%']:.1%}")
print(f"  min:     {fg_summary['min']:.1%}")
print(f"  max:     {fg_summary['max']:.1%}")

# Flag any images that look pathological — very small or very large
# masks. These don't fail the section, but they're worth eyeballing.
# The formal pass/fail decision is 2.3's IoU validation, not this.
EXTREME_LOW = 0.02   # below 2% — SAM probably found nothing
EXTREME_HIGH = 0.95  # above 95% — SAM probably found everything

extreme_low = leaf_masks_metadata[
    leaf_masks_metadata["fg_fraction"] < EXTREME_LOW
]
extreme_high = leaf_masks_metadata[
    leaf_masks_metadata["fg_fraction"] > EXTREME_HIGH
]

n_low = len(extreme_low)
n_high = len(extreme_high)

if n_low or n_high:
    print(f"\nExtreme masks worth eyeballing:")
    print(f"  {n_low} image(s) with fg_fraction < {EXTREME_LOW:.0%}")
    print(f"  {n_high} image(s) with fg_fraction > {EXTREME_HIGH:.0%}")
    print(
        f"  These don't fail Section 2 — 2.3's IoU check is the "
        f"formal validation — but extreme outliers can change "
        f"what 2.4 routes to the fallback."
    )
else:
    print(
        f"\nNo extreme masks "
        f"(none below {EXTREME_LOW:.0%} or above {EXTREME_HIGH:.0%})."
    )

## 4 — Check SAM segmentation quality

Section 3 produced 81 raw SAM masks. Before those masks get used by the categorization pipeline in Section 6, this section answers one question:

**Is SAM reliable enough to trust?** N01 committed to a sanity check (not a precision estimate) of SAM's segmentation. This section executes that sanity check in two parts:

| Sub | What it does |
|---|---|
| 4.1 | Eyeball 10 images sampled across the fg_fraction range — visual check of SAM's boundaries |
| 4.2 | If the 4.1 check reveals a failure pattern at fg extremes, inspect the borderline band to pin the threshold |

Section 5 then acts on what the visual checks revealed.

This section diverges from N01's locked precommit, which called for hand-segmented reference masks and an IoU comparison. The visual-check substitute is defensible because (a) N01 framed the validation as a sanity check, not a precision estimate, and a 10-image qualitative inspection delivers the same kind of evidence; (b) hand-traced boundaries are systematically less tight than SAM's, making IoU values hard to interpret as pass/fail; (c) the working set is small enough (81 images) that follow-up inspection is feasible if a pattern emerges. The deviation is documented here and should be reflected in N01's precommit before the program ships.

## 4.1 — Visual check of SAM masks

You committed in N01 to validating SAM's segmentation as a *sanity check, not a precision estimate*. The original procedure was to hand-segment five images and compute IoU against the corresponding SAM masks. This section executes the sanity check in a simpler form: a 10-image qualitative grid. You'll eyeball SAM's output across the fg_fraction range — high, low, middle, and random — and decide whether SAM is reliable enough on this working set to ground the predicate computations that follow.

Three reasons the visual check is a defensible substitute for the IoU procedure:

1. **The validation's stated purpose was a sanity check, not a precision estimate.** A visual inspection of 10 images delivers the same kind of evidence (SAM either does or doesn't reliably find the leaf) without the precision-asymmetry problem that the IoU procedure runs into — a hand-traced boundary is typically less tight than SAM's, so even agreement-with-SAM lands in the 0.85–0.92 IoU range and is hard to interpret as pass/fail.

2. **SAM is the field-standard prompt-driven segmenter for natural images.** Its behavior on plant imagery is documented in prior work; the failure modes it has on PD are likely to be the obvious ones (off-center leaf, multiple leaves at similar depth, severely occluded leaf), all of which a visual scan surfaces directly.

3. **The working set is small.** 81 images. If the 10-image sample reveals a systematic problem, you have the option to eyeball the full set. The IoU procedure on 5 images doesn't offer that level of follow-through.

What to look for in the grid below:

- **Did SAM find the leaf?** The red contour should trace the leaf boundary, not a shadow, a background object, or a different leaf nearby.
- **Did SAM grab the *whole* leaf?** Missed leaflets, truncated edges, or holes inside the mask all matter for Section 3's predicates.
- **Did SAM grab *only* the leaf?** Spillover into stems, background foliage, or hands holding the leaf will inflate the "in-leaf" attention mass downstream.

If SAM looks reliable across the sample, document that in the notes cell below and proceed to Section 3. If you see a recurring failure mode, note it — Section 3's predicates can still be computed, but the failure pattern is something the paper's discussion section needs to acknowledge.

In [ ]:
### Visual check of SAM masks — 10 images sampled across fg_fraction

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Where to save the visual-check figure for the methods record.
VISUAL_CHECK_PATH = OUTPUT_DIR / "sam_visual_check.png"

# Stratified sample: 2 highest fg_fraction, 2 lowest, 2 nearest the median,
# plus 4 random. SEED=12 matches the seed used elsewhere in R2-Q2 for
# sample reproducibility.
SEED = 12
sorted_meta = leaf_masks_metadata.sort_values("fg_fraction").reset_index(drop=True)
n = len(sorted_meta)
mid = n // 2

fixed = sorted_meta.iloc[[n - 1, n - 2, 0, 1, mid - 1, mid]]
remaining = sorted_meta[~sorted_meta["filename"].isin(fixed["filename"])]
random_picks = remaining.sample(4, random_state=SEED)

visual_check_picks = (
    pd.concat([fixed, random_picks])
    .merge(working_set[["filename", "host", "true_category"]], on="filename", how="left")
    .reset_index(drop=True)
)

print(f"Visual check sample (SEED={SEED}, n={len(visual_check_picks)}):")
print(visual_check_picks[["filename", "host", "true_category", "fg_fraction"]].to_string(index=False))

# 5 rows × 2 columns. Source image with SAM mask boundary drawn in red.
fig, axes = plt.subplots(5, 2, figsize=(14, 28))
axes = axes.flatten()

for ax, (_, row) in zip(axes, visual_check_picks.iterrows()):
    image_np = np.array(load_pd_image(row["filename"]))
    mask = np.array(Image.open(row["mask_path"]))

    ax.imshow(image_np)
    ax.contour(mask > 0, levels=[0.5], colors="red", linewidths=1.8)
    ax.axis("off")

    short_name = row["filename"] if len(row["filename"]) <= 50 else row["filename"][:47] + "..."
    ax.set_title(
        f"{row['host']} · {row['true_category']} · fg={row['fg_fraction']:.0%}\n"
        f"{short_name}",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig(VISUAL_CHECK_PATH, dpi=100, bbox_inches="tight")
plt.show()
print(f"\nSaved {VISUAL_CHECK_PATH}")

## 4.2 — Borderline threshold check

The 10-image check above revealed a clear pattern: SAM's failures clustered at the extremes of the foreground-fraction distribution. Images at `fg < 0.05` were catastrophically undersegmented (SAM grabbed a tiny region — a lesion, a bud — and called nothing else leaf). Images at `fg > 0.85` were undersegmented in the opposite direction (SAM grabbed everything, unable to find a leaf edge in a multi-leaf or low-contrast scene). The middle of the distribution was reliable.

The diagnostic count below confirms this is workable: 11 images at `fg < 0.05`, 2 at `fg > 0.85`, leaving 68 of 81 (84%) retained in the band `0.05 ≤ fg ≤ 0.85`.

What's still unsettled is where exactly the lower edge of "reliable" sits. The 10-image check happened to include image #8 at `fg = 0.10` (a diseased tomato leaflet, clean segmentation) and image #10 at `fg = 0.05` (a sliver of one leaf in a multi-leaf stock photo, borderline failure). That sample alone doesn't say whether the band `0.05 ≤ fg < 0.10` is mostly clean or mostly degenerate.

This subsection settles that. The 9 images sitting in `0.05 ≤ fg < 0.10` are shown below with their SAM masks overlaid, in the same red-contour style as 4.1. If most are clean, the filter locks at `fg ≥ 0.05`. If most are degenerate, the filter tightens to `fg ≥ 0.10`.

In [ ]:
### Borderline visual check — the 9 images in 0.05 ≤ fg < 0.10

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

BORDERLINE_CHECK_PATH = OUTPUT_DIR / "sam_borderline_check.png"

# Pull the borderline band and join the working-set context columns.
borderline_picks = (
    leaf_masks_metadata[
        (leaf_masks_metadata["fg_fraction"] >= 0.05)
        & (leaf_masks_metadata["fg_fraction"] < 0.10)
    ]
    .sort_values("fg_fraction")
    .merge(working_set[["filename", "host", "true_category"]], on="filename", how="left")
    .reset_index(drop=True)
)

print(f"Borderline band (0.05 ≤ fg < 0.10): {len(borderline_picks)} images")
print(borderline_picks[["filename", "host", "true_category", "fg_fraction"]].to_string(index=False))

# 5 rows × 2 columns gives 10 panels for 9 images. The last panel is
# hidden so the layout doesn't show an empty axis.
fig, axes = plt.subplots(5, 2, figsize=(14, 28))
axes = axes.flatten()

for ax, (_, row) in zip(axes, borderline_picks.iterrows()):
    image_np = np.array(load_pd_image(row["filename"]))
    mask = np.array(Image.open(row["mask_path"]))

    ax.imshow(image_np)
    ax.contour(mask > 0, levels=[0.5], colors="red", linewidths=1.8)
    ax.axis("off")

    short_name = row["filename"] if len(row["filename"]) <= 50 else row["filename"][:47] + "..."
    ax.set_title(
        f"{row['host']} · {row['true_category']} · fg={row['fg_fraction']:.1%}\n"
        f"{short_name}",
        fontsize=9,
    )

# Hide unused panels.
for ax in axes[len(borderline_picks):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(BORDERLINE_CHECK_PATH, dpi=100, bbox_inches="tight")
plt.show()
print(f"\nSaved {BORDERLINE_CHECK_PATH}")

## 5 — Filter the working set

Sections 4.1 and 4.2 established that SAM produces reliable masks when `0.05 ≤ fg_fraction ≤ 0.85` and unreliable masks outside that band. This section applies the filter as a working-set decision: the analysis from Section 6 onward runs on the filtered subset, not on the original 81 images.

**Why a filter, not a fallback.** N01's locked precommit named "color-based thresholding" as the fallback for images where SAM failed. The visual evidence in 4.1 and 4.2 changed our understanding of what "SAM failing" actually means here. The 13 dropped images aren't cases where SAM picked the wrong segmentation method — they're cases where no center-point segmentation method could reliably do the right thing because the image itself doesn't have a single dominant leaf foreground. Wide field shots, multi-leaf compositions, low-contrast foliage-on-foliage scenes. A color-based fallback would face the same ill-posedness; it wouldn't recover information the image doesn't contain.

**What the filter costs.** Going from 81 to 68 images is a 16% reduction. The taxonomy distribution in Section 8 will be computed on the smaller set. Per-host and per-category breakdowns become noisier (some cells will be small), and the paper's discussion section will need to note that the analysis was restricted to images where the segmentation problem was well-posed.

**What the filter preserves.** The full 81-row table stays on disk as `working_set_unfiltered.parquet`. Any later question — "what would the result look like on the full set?" or "did we drop a disproportionate share of one disease category?" — can be answered without re-running anything upstream.

**Forward-pointer to the precommit.** N01's `fallback_if_sam_fails` field doesn't match what this section actually does. That's a precommit edit to make before the program ships, captured as a separate task; it doesn't block N04's analysis.

In [ ]:
### Filter the working set to images where SAM produced a reliable mask

# Bounds settled in 4.1 and 4.2.
FG_LOWER = 0.05
FG_UPPER = 0.85

# Preserve the unfiltered table for forensic comparison.
working_set_unfiltered = working_set.copy()
unfiltered_path = OUTPUT_DIR / "working_set_unfiltered.parquet"
working_set_unfiltered.to_parquet(unfiltered_path)

# Build the filter mask against leaf_masks_metadata (the source of truth
# for fg_fraction per filename) and apply to both the working set and
# the metadata table. Downstream sections will see filtered versions of
# both.
reliable_mask = (
    (leaf_masks_metadata["fg_fraction"] >= FG_LOWER)
    & (leaf_masks_metadata["fg_fraction"] <= FG_UPPER)
)
reliable_filenames = set(leaf_masks_metadata.loc[reliable_mask, "filename"])

n_before = len(working_set)
working_set = working_set[working_set["filename"].isin(reliable_filenames)].reset_index(drop=True)
leaf_masks_metadata = leaf_masks_metadata[reliable_mask].reset_index(drop=True)
n_after = len(working_set)

# Persist the filtered working set as the canonical analysis-ready table.
analysis_ready_path = OUTPUT_DIR / "analysis_ready.parquet"
working_set.to_parquet(analysis_ready_path)

# Surface the per-host and per-category breakdown so any disproportionate
# drops show up immediately rather than turning up as a surprise in
# Section 8.
print(f"Filter bounds: {FG_LOWER:.2f} ≤ fg_fraction ≤ {FG_UPPER:.2f}")
print(f"Working set: {n_before} → {n_after} images ({n_after / n_before:.0%} retained)")
print(f"  dropped: {n_before - n_after}")
print()
print(f"Saved {analysis_ready_path}")
print(f"Saved {unfiltered_path}  (forensic reference)")
print()

dropped = working_set_unfiltered[~working_set_unfiltered["filename"].isin(reliable_filenames)]
print("Per-host dropped count:")
print(dropped["host"].value_counts().to_string())
print()
print("Per-category dropped count:")
print(dropped["true_category"].value_counts().to_string())

## 6 — Compute the five derived quantities

Section 5 left you with 68 working-set images, each paired with a SAM mask (in `leaf_masks_metadata`) and a Grad-CAM heatmap (in `gradcam_metadata`). This section computes five per-image quantities that the predicates in Section 7 will threshold against. All five are defined relative to the leaf mask — that's the whole reason Sections 2–5 needed to land first.

The five quantities, all defined per image:

| Quantity | What it measures | Precommit threshold (Section 7) |
|---|---|---|
| `m_out` | Fraction of attention mass *outside* the leaf | `m_out > 0.5` → background_attended |
| `m_in` | Fraction of attention mass *inside* the leaf | `m_in ≥ 0.5` (gate for symptom predicate) |
| `m_perimeter` | Fraction of attention mass in a band hugging the leaf edge | `m_perimeter > 0.3` → leaf_shape_attended |
| `concentration` | Fraction of in-leaf attention captured by the smallest region containing 80% of in-leaf mass | `concentration > 0.6` (with `m_in ≥ 0.5`) → symptom_attended_but_wrong_class |
| `lighting_correlation` | Spearman ρ between the heatmap and a pixelwise brightness map | `|ρ| > 0.5` → lighting_attended |

A few decisions baked into the code below — defaults consistent with the precommit and standard practice, called out here so they're visible:

**Heatmap upsampling: bilinear.** N03's heatmaps are 7×7 (the spatial resolution of the last conv block in ResNet-50). The precommit says to upsample to the image's native resolution before any mask comparison. Bilinear is the standard choice for Grad-CAM visualization and avoids the blocky artifacts of nearest-neighbor.

**Attention mass normalization: ReLU then sum-to-one.** Grad-CAM produces signed values; the precommit pins the normalization to "ReLU then normalize to sum to one." A heatmap that's all-zeros after ReLU (rare but possible) gets uniform mass as a degenerate fallback — flagged in the output so any such cases surface rather than silently producing NaN downstream.

**Perimeter band: dilate-by-8 minus erode-by-8.** Pulled verbatim from the precommit. A 16-pixel-wide band hugging the leaf edge. Wide enough to catch attention that's "on the silhouette" without being so wide that it overlaps the leaf interior.

**Concentration: top-down thresholding.** Lower the heatmap threshold until the surviving in-leaf pixels capture 80% of in-leaf mass; take the largest connected component at that threshold; the ratio of its area to the leaf area is concentration. The precommit's phrasing ("smallest contiguous region capturing 80% of in-leaf mass") describes this procedure.

**Brightness for the lighting correlation: luminance.** `0.299 R + 0.587 G + 0.114 B` (BT.601 weights), computed on the original image at native resolution. The Spearman correlation runs over all pixels in the image (not just in-leaf) because the lighting hypothesis is that the model attends to bright/dark *regions of the image*, not bright/dark parts of the leaf specifically.

After this section runs, `derived_quantities.parquet` has 68 rows and one column per quantity plus a degenerate-mass flag. Section 7 reads it and applies the four predicates.

In [ ]:
### 6 — Define the five per-image quantity functions

import numpy as np
from PIL import Image
from scipy import ndimage
from scipy.stats import spearmanr
from skimage.transform import resize


def upsample_heatmap(heatmap_7x7, target_hw):
    """Upsample a 7×7 Grad-CAM heatmap to (H, W) with bilinear interpolation.

    target_hw : tuple
        (H, W) — the source image's native resolution.
    """
    return resize(heatmap_7x7, target_hw, order=1, mode="edge", anti_aliasing=False)


def normalize_attention(heatmap):
    """Apply ReLU then normalize to sum to 1. Returns (normalized_heatmap, is_degenerate).

    A heatmap that's all-zero after ReLU gets uniform mass as a fallback. The
    is_degenerate flag lets the caller surface these cases rather than swallow
    them silently.
    """
    relu = np.maximum(heatmap, 0)
    total = relu.sum()
    if total == 0:
        H, W = heatmap.shape
        return np.full_like(relu, 1.0 / (H * W)), True
    return relu / total, False


def compute_perimeter_band(mask, width_px=8):
    """Band hugging the mask boundary: dilate(mask, w) minus erode(mask, w).

    Returns a boolean mask of the same shape as the input. Width is the
    precommit's value (8 px each side, 16 px total band thickness).
    """
    structure = ndimage.generate_binary_structure(2, 1)
    dilated = ndimage.binary_dilation(mask, structure=structure, iterations=width_px)
    eroded = ndimage.binary_erosion(mask, structure=structure, iterations=width_px)
    return dilated & ~eroded


def compute_concentration(heatmap_norm, leaf_mask):
    """Concentration: how compact the 80%-mass region is, relative to the leaf.

    Range: [0, 1]. Near 1 = attention is focused in a small region.
    Near 0 = attention is spread across most of the leaf.

    The procedure follows the precommit's phrasing:
      1. Restrict the heatmap to in-leaf pixels.
      2. Lower a threshold from the maximum value down; at each threshold,
         the surviving pixels are those above it.
      3. Stop when the surviving pixels cover >=80% of the in-leaf mass.
      4. Take the largest connected component of the surviving region.
      5. Return 1 - area(component) / area(leaf_mask).

    The 1 - ratio is what flips a small region (focused) to a value near 1
    and a large region (diffuse) to a value near 0, matching the predicate
    `concentration > 0.6` in N01 Section 8.4.

    If the leaf mask has no in-leaf attention at all (in-leaf mass = 0),
    concentration is 0 by convention (not focused on anything).
    """
    in_leaf_mass = heatmap_norm[leaf_mask].sum()
    if in_leaf_mass == 0:
        return 0.0

    # Sort in-leaf pixel intensities descending; find the threshold at
    # which cumulative mass first hits 80% of in-leaf mass.
    in_leaf_values = heatmap_norm[leaf_mask]
    sorted_desc = np.sort(in_leaf_values)[::-1]
    cumulative = np.cumsum(sorted_desc)
    target = 0.80 * in_leaf_mass
    threshold_idx = np.searchsorted(cumulative, target)
    threshold = sorted_desc[min(threshold_idx, len(sorted_desc) - 1)]

    # Surviving region: in-leaf pixels at or above the threshold.
    surviving = leaf_mask & (heatmap_norm >= threshold)

    # Largest connected component of the surviving region.
    labels, n_components = ndimage.label(surviving)
    if n_components == 0:
        return 0.0
    component_sizes = ndimage.sum(surviving, labels, range(1, n_components + 1))
    largest_size = component_sizes.max()

    return float(1.0 - largest_size / leaf_mask.sum())


def compute_brightness(image_rgb):
    """Per-pixel luminance using BT.601 weights. Returns float array shape (H, W)."""
    return 0.299 * image_rgb[..., 0] + 0.587 * image_rgb[..., 1] + 0.114 * image_rgb[..., 2]


def compute_quantities(filename):
    """Compute the five derived quantities for one image.

    Returns a dict with: m_in, m_out, m_perimeter, concentration,
    lighting_correlation, degenerate_attention.
    """
    # Load the source image at native resolution (RGB, uint8).
    image_pil = load_pd_image(filename)
    image_np = np.array(image_pil)
    H, W = image_np.shape[:2]

    # Load and normalize the Grad-CAM heatmap.
    heatmap_path = OUTPUT_DIR / gradcam_metadata.set_index("filename").loc[filename, "heatmap_path"]
    heatmap_7x7 = np.load(heatmap_path)
    heatmap_full = upsample_heatmap(heatmap_7x7, (H, W))
    heatmap_norm, is_degenerate = normalize_attention(heatmap_full)

    # Load the SAM mask. Mode 'L' → bool via > 0.
    mask_path = leaf_masks_metadata.set_index("filename").loc[filename, "mask_path"]
    leaf_mask = np.array(Image.open(mask_path)) > 0

    # The mask and heatmap must share shape — both should be at native
    # resolution, but a shape mismatch here would silently produce wrong
    # attention masses, so surface it.
    if leaf_mask.shape != heatmap_norm.shape:
        raise ValueError(
            f"{filename}: mask shape {leaf_mask.shape} ≠ heatmap shape "
            f"{heatmap_norm.shape}. One of them is at the wrong resolution."
        )

    # Quantity 1 and 2: in-leaf and out-of-leaf attention mass.
    m_in = float(heatmap_norm[leaf_mask].sum())
    m_out = float(heatmap_norm[~leaf_mask].sum())

    # Quantity 3: perimeter-band attention mass.
    band = compute_perimeter_band(leaf_mask, width_px=8)
    m_perimeter = float(heatmap_norm[band].sum())

    # Quantity 4: concentration of in-leaf attention.
    concentration = compute_concentration(heatmap_norm, leaf_mask)

    # Quantity 5: Spearman ρ between heatmap and pixelwise brightness.
    brightness = compute_brightness(image_np)
    rho, _ = spearmanr(heatmap_norm.ravel(), brightness.ravel())
    lighting_correlation = float(rho)

    return {
        "filename": filename,
        "m_in": m_in,
        "m_out": m_out,
        "m_perimeter": m_perimeter,
        "concentration": concentration,
        "lighting_correlation": lighting_correlation,
        "degenerate_attention": is_degenerate,
    }


print("Quantity functions defined: m_in, m_out, m_perimeter, concentration, lighting_correlation")

In [ ]:
### 6 — Compute derived quantities for every retained working-set image

from tqdm.auto import tqdm
import time

start_time = time.time()

records = []
for filename in tqdm(working_set["filename"], desc="Deriving"):
    records.append(compute_quantities(filename))

derived_quantities = pd.DataFrame(records)

elapsed = time.time() - start_time

# Persist for Section 7.
derived_path = OUTPUT_DIR / "derived_quantities.parquet"
derived_quantities.to_parquet(derived_path)

# Summary surfaces both the run-level stats and any per-image red flags.
print(f"\nDerived quantities for {len(derived_quantities)} images in {elapsed:.1f}s")
print(f"  ({elapsed / len(derived_quantities):.2f}s per image average)")
print(f"  Saved to: {derived_path}\n")

# Quick consistency check: m_in + m_out should be ≈ 1.0 for every image
# (the heatmap was normalized to sum to 1, and every pixel is either
# in-leaf or out-of-leaf).
total_mass = derived_quantities["m_in"] + derived_quantities["m_out"]
mass_off_by = (total_mass - 1.0).abs().max()
print(f"Mass conservation check (m_in + m_out ≈ 1):")
print(f"  max deviation from 1.0: {mass_off_by:.2e}")

# Flag degenerate cases.
n_degenerate = derived_quantities["degenerate_attention"].sum()
if n_degenerate > 0:
    print(f"\n⚠ {n_degenerate} image(s) had all-zero attention after ReLU "
          f"and were assigned uniform mass. Affected filenames:")
    for fn in derived_quantities.loc[derived_quantities["degenerate_attention"], "filename"]:
        print(f"    - {fn}")
else:
    print(f"\nNo degenerate-attention cases.")

# Quick descriptive stats — a sanity check on the value ranges before
# Section 7 starts thresholding against them.
print("\nDistribution of derived quantities:")
print(derived_quantities[
    ["m_in", "m_out", "m_perimeter", "concentration", "lighting_correlation"]
].describe().round(3).to_string())

## 7 — Apply the four predicates

Section 6 produced five derived quantities for each of the 68 retained images. This section turns those quantities into category assignments by applying the four predicates from the precommit in their committed first-match-wins order.

### The four predicates, in order

Per the precommit's `categorization_procedure.predicate_order`:

| # | Predicate | Fires when |
|---|---|---|
| 1 | `symptom_attended_but_wrong_class` | `m_in ≥ 0.5` AND `concentration > 0.6` |
| 2 | `background_attended` | `m_out > 0.5` |
| 3 | `lighting_attended` | `\|lighting_correlation\| > 0.5` |
| 4 | `leaf_shape_attended` | `m_perimeter > 0.3` |

The precommit's `predicate_scheme` is **first-match-wins, multi-fires route to other**. For each image:

1. Test predicate 1. If it fires, the image is assigned that category and the rest are still evaluated (to detect multi-fires for the audit trail). If exactly one predicate fires across all four, the first-match category stands.
2. If zero predicates fire, the image is assigned `other_ambiguous`.
3. If two or more predicates fire, the image is assigned `other_ambiguous` and the multi-fire is recorded.

This means `other_ambiguous` has two operationally distinct flavors: "no predicate could classify this" and "multiple predicates fit, the precommit doesn't say which wins." Section 8 surfaces the distinction so the methods write-up can be honest about it.

### Why the predicate functions are hardcoded, not read from the precommit

The N04 structure handoff specified this explicitly. Predicate logic is part of the operational definition of each failure mode, not a tunable parameter. If the predicate definitions ever need to change, that's a precommit revision (with the appropriate documentation), not a runtime configuration change. Hardcoding makes the precommit ↔ code correspondence visible in the source.

Each predicate function below cites the precommit field it implements.

### Why predicate order is locked

`symptom_attended_but_wrong_class` is the predicate the program *most wants to detect* — it's the category that addresses R2-Q2's central question ("when the model gets it wrong, is it because it's looking at the right thing but interpreting it wrong, or because it's looking at the wrong thing entirely?"). Putting it first means an image that satisfies both the symptom predicate and one of the others (most likely `background_attended` via `m_out > 0.5`) gets credit for the symptom diagnosis rather than being silently overwritten by a less informative category.

The remaining order — `background_attended`, `lighting_attended`, `leaf_shape_attended` — runs roughly from "model attended somewhere obviously wrong" to "model attended somewhere subtle." The exact ordering past position 1 matters less because the multi-fire detection catches anything ambiguous and routes it to `other_ambiguous` regardless.

In [ ]:
### 7 — Define the four predicates

# Each function takes a row (a pandas Series with the five derived quantity
# columns) and returns a bool. Thresholds are hardcoded per the precommit,
# with the precommit field cited in each docstring.


def fires_symptom_attended_but_wrong_class(row):
    """precommit.categorization_procedure.thresholds.T4_symptom_attended_but_wrong_class

    Fires when in-leaf mass is at least 0.5 AND concentration exceeds 0.6.

    Meaning: the model concentrated its attention on a small in-leaf region
    (a candidate symptom location), but produced the wrong class label.
    """
    return (row["m_in"] >= 0.5) and (row["concentration"] > 0.6)


def fires_background_attended(row):
    """precommit.categorization_procedure.thresholds.T1_background_attended

    Fires when out-of-leaf mass exceeds 0.5.

    Meaning: the model's attention fell predominantly outside the leaf.
    """
    return row["m_out"] > 0.5


def fires_lighting_attended(row):
    """precommit.categorization_procedure.thresholds.T3_lighting_attended

    Fires when |Spearman ρ between heatmap and brightness| exceeds 0.5.

    Meaning: the model's attention tracked brightness more than it tracked
    plant content — either bright spots (positive ρ) or dark spots (negative ρ).
    """
    return abs(row["lighting_correlation"]) > 0.5


def fires_leaf_shape_attended(row):
    """precommit.categorization_procedure.thresholds.T2_leaf_shape_attended

    Fires when perimeter-band mass exceeds 0.3.

    Meaning: at least 30% of the model's attention fell within the 16-pixel
    band hugging the leaf's edge — suggesting attention to shape/silhouette
    rather than to interior content.
    """
    return row["m_perimeter"] > 0.3


# The pre-committed predicate order. The list itself is locked by the
# precommit's predicate_order field; this is the operational form.
PREDICATE_ORDER = [
    ("symptom_attended_but_wrong_class", fires_symptom_attended_but_wrong_class),
    ("background_attended", fires_background_attended),
    ("lighting_attended", fires_lighting_attended),
    ("leaf_shape_attended", fires_leaf_shape_attended),
]


def assign_category(row):
    """Apply the four predicates to one row of derived_quantities.

    Returns a dict with:
      - one boolean per predicate (which fired)
      - multi_fire_predicates: list of predicate names that fired (0..4 entries)
      - assigned_category: the precommit's first-match-wins choice, or
                           "other_ambiguous" for 0-fire or multi-fire cases.
    """
    fired = []
    fire_flags = {}
    for name, predicate in PREDICATE_ORDER:
        result = predicate(row)
        fire_flags[f"fires_{name}"] = result
        if result:
            fired.append(name)

    if len(fired) == 1:
        assigned = fired[0]
    else:
        # Zero fires or multiple fires both route to other_ambiguous.
        assigned = "other_ambiguous"

    return {
        **fire_flags,
        "multi_fire_predicates": fired,
        "assigned_category": assigned,
    }


print("Predicate functions defined in pre-committed order:")
for name, _ in PREDICATE_ORDER:
    print(f"  {name}")

In [ ]:
### 7 — Apply the predicates to all 68 retained images

# Build the per-row result table.
predicate_records = []
for _, row in derived_quantities.iterrows():
    result = assign_category(row)
    result["filename"] = row["filename"]
    predicate_records.append(result)

predicate_results = pd.DataFrame(predicate_records)

# Column order: filename first, then the four firing booleans in pre-committed
# order, then the multi-fire list, then the final category. Matches the
# table you'd read top-to-bottom: identifier → evidence → audit → verdict.
predicate_results = predicate_results[
    ["filename"]
    + [f"fires_{name}" for name, _ in PREDICATE_ORDER]
    + ["multi_fire_predicates", "assigned_category"]
]

# Persist for Section 8.
predicate_path = OUTPUT_DIR / "predicate_results.parquet"
predicate_results.to_parquet(predicate_path)
print(f"Saved {predicate_path}\n")

# Surface the distribution. Section 8 produces the final figure and JSON;
# this print is a sanity check that the predicate logic is doing something
# reasonable before that step.
print(f"Category counts (n={len(predicate_results)}):")
category_counts = predicate_results["assigned_category"].value_counts()
for category, count in category_counts.items():
    print(f"  {count:>3}  {category}")
print()

# Distinguish the two flavors of "other_ambiguous" — zero-fires vs multi-fires —
# because they have different methodological implications. Zero-fires say
# "the predicate net has gaps"; multi-fires say "the predicates overlap and
# the precommit didn't break the tie."
other_mask = predicate_results["assigned_category"] == "other_ambiguous"
other_rows = predicate_results[other_mask]
n_zero_fire = (other_rows["multi_fire_predicates"].str.len() == 0).sum()
n_multi_fire = (other_rows["multi_fire_predicates"].str.len() >= 2).sum()
print(f"Breakdown of other_ambiguous (n={other_mask.sum()}):")
print(f"  zero-fire (no predicate matched): {n_zero_fire}")
print(f"  multi-fire (≥2 predicates matched): {n_multi_fire}")

# Per-predicate fire counts — useful for catching threshold-calibration
# issues (a predicate that fires on 1/68 images may be calibrated too
# strictly; one that fires on 60/68 may be calibrated too loosely).
print(f"\nPer-predicate fire counts (independent of first-match-wins):")
for name, _ in PREDICATE_ORDER:
    n_fired = predicate_results[f"fires_{name}"].sum()
    print(f"  {n_fired:>3}  {name}")

# Multi-fire pattern detail — which combinations of predicates co-fired,
# useful for spotting predicate-pair overlaps that the precommit's ordering
# might want to address in a future revision.
multi_fire_rows = predicate_results[predicate_results["multi_fire_predicates"].str.len() >= 2]
if len(multi_fire_rows) > 0:
    print(f"\nMulti-fire patterns ({len(multi_fire_rows)} rows):")
    pattern_counts = (
        multi_fire_rows["multi_fire_predicates"]
        .apply(lambda lst: " + ".join(lst))
        .value_counts()
    )
    for pattern, count in pattern_counts.items():
        print(f"  {count:>3}  {pattern}")
else:
    print(f"\nNo multi-fire cases.")

## 8 — The failure-mode distribution

This section ties the work of Sections 1–7 into the form the paper needs. Three deliverables:

| File | What it is |
|---|---|
| `categorizations.parquet` | Per-image audit table — one row per analyzed image, all derived quantities, all predicate firings, the assigned category. The full evidence trail behind every category assignment. |
| `taxonomy_distribution.json` | Reader-facing summary — results, methodology recap, interpretation. Same three-section shape as R2-Q1's deliverables. |
| `taxonomy_distribution.png` | The five-bar chart for the paper — four named categories sorted by count descending, `other_ambiguous` anchored at the right with a visually distinct color. |

The precommit's `aggregation_level` field is `overall_taxonomy_distribution`, so this section produces the overall distribution and stops there. Per-host and per-category breakdowns are not part of R2-Q2's finding — they would be hypothesis-generating, not finding-producing, and the small per-cell counts (some host × disease cells have ≤ 5 images in the working set) would carry wide enough error bars to be uninterpretable.

The interpretation field in the JSON is a placeholder. Like the precommit's `reasoning` fields, it expects student-rewritten text that addresses what the distribution actually says.

In [ ]:
### 8 — Assemble categorizations.parquet

# Join the three per-image tables on filename. All three are filename-keyed
# and aligned to the 68 retained images.
categorizations = (
    derived_quantities
    .merge(predicate_results, on="filename", how="inner")
)

# Column order: identifier → derived quantities → predicate firings →
# multi-fire audit → final category. Matches the audit-trail reading order.
categorizations = categorizations[
    ["filename"]
    + ["m_in", "m_out", "m_perimeter", "concentration", "lighting_correlation"]
    + ["degenerate_attention"]
    + [f"fires_{name}" for name, _ in PREDICATE_ORDER]
    + ["multi_fire_predicates", "assigned_category"]
]

# Persist.
categorizations_path = OUTPUT_DIR / "categorizations.parquet"
categorizations.to_parquet(categorizations_path)
print(f"Saved {categorizations_path}")
print(f"  shape: {categorizations.shape}")
print(f"  columns: {list(categorizations.columns)}")

# Spot-check: row count must equal the analyzed working set.
assert len(categorizations) == len(working_set), (
    f"Row count mismatch: categorizations has {len(categorizations)} rows, "
    f"working set has {len(working_set)}"
)
print(f"\nRow count matches working set: {len(categorizations)}")

In [ ]:
### 8 — Synthesize taxonomy_distribution.json and taxonomy_distribution.png

import json
import matplotlib.pyplot as plt

# ----- Results section -----
# All five categories must appear in the JSON even if their count is zero,
# so any reader can confirm the predicate fired (or didn't) on this run.
ALL_CATEGORIES = [name for name, _ in PREDICATE_ORDER] + ["other_ambiguous"]

category_counts = {
    cat: int((categorizations["assigned_category"] == cat).sum())
    for cat in ALL_CATEGORIES
}
n_analyzed = len(categorizations)
category_proportions = {
    cat: round(count / n_analyzed, 4)
    for cat, count in category_counts.items()
}

# other_ambiguous breakdown — distinguishes zero-fire from multi-fire.
other_mask = categorizations["assigned_category"] == "other_ambiguous"
other_rows = categorizations[other_mask]
n_zero_fire = int((other_rows["multi_fire_predicates"].str.len() == 0).sum())
n_multi_fire = int((other_rows["multi_fire_predicates"].str.len() >= 2).sum())

# Multi-fire pattern detail.
multi_fire_rows = categorizations[categorizations["multi_fire_predicates"].str.len() >= 2]
multi_fire_patterns = (
    multi_fire_rows["multi_fire_predicates"]
    .apply(lambda lst: " + ".join(lst))
    .value_counts()
    .to_dict()
)
multi_fire_patterns = {k: int(v) for k, v in multi_fire_patterns.items()}

# Per-predicate fire counts — independent of first-match-wins assignment.
per_predicate_fires = {
    name: int(categorizations[f"fires_{name}"].sum())
    for name, _ in PREDICATE_ORDER
}

# Working-set filter provenance — what got dropped before this analysis ran.
n_unfiltered = len(working_set_unfiltered)
n_filtered_out = n_unfiltered - n_analyzed

results = {
    "n_misclassifications_analyzed": n_analyzed,
    "n_misclassifications_unfiltered": n_unfiltered,
    "n_filtered_out_for_unreliable_segmentation": n_filtered_out,
    "filter_bounds_fg_fraction": [FG_LOWER, FG_UPPER],
    "category_counts": category_counts,
    "category_proportions": category_proportions,
    "other_ambiguous_breakdown": {
        "zero_fire": n_zero_fire,
        "multi_fire": n_multi_fire,
    },
    "multi_fire_patterns": multi_fire_patterns,
    "per_predicate_fire_counts": per_predicate_fires,
    "n_degenerate_attention": int(categorizations["degenerate_attention"].sum()),
}

# ----- Methodology section -----
methodology = {
    "segmentation": (
        "SAM (Segment Anything Model) with single foreground point at image center. "
        "Working set filtered to fg_fraction in [0.05, 0.85] to exclude images "
        "where the segmentation problem was ill-posed (no dominant leaf foreground)."
    ),
    "attention_normalization": "ReLU then normalize to sum to one",
    "perimeter_band_definition": "leaf mask dilated by 8 pixels minus leaf mask eroded by 8 pixels",
    "concentration_definition": (
        "fraction of in-leaf attention captured by the smallest contiguous "
        "region containing 80% of in-leaf mass"
    ),
    "lighting_correlation": (
        "Spearman ρ between Grad-CAM heatmap and pixelwise BT.601 luminance "
        "(0.299·R + 0.587·G + 0.114·B)"
    ),
    "predicate_thresholds": {
        "T1_background_attended": "m_out > 0.5",
        "T2_leaf_shape_attended": "m_perimeter > 0.3",
        "T3_lighting_attended": "|lighting_correlation| > 0.5",
        "T4_symptom_attended_but_wrong_class": "m_in ≥ 0.5 AND concentration > 0.6",
    },
    "predicate_scheme": "first-match-wins; zero-fire and multi-fire both route to other_ambiguous",
    "predicate_order": [name for name, _ in PREDICATE_ORDER],
    "aggregation_level": "overall_taxonomy_distribution (no per-host or per-category breakdowns)",
}

# ----- Interpretation placeholder -----
interpretation = (
    "PLACEHOLDER — REWRITE BEFORE SUBMISSION. The student-authored interpretation "
    "should address, at minimum: (1) the dominant category in `category_counts` "
    "and what it implies about the model's failure mode; (2) the count of "
    "`symptom_attended_but_wrong_class` and what its prevalence (or rarity) says "
    "about whether misclassifications are 'right place, wrong label' vs 'wrong "
    "place entirely'; (3) any predicate that fired frequently but rarely won "
    "first-match (see `per_predicate_fire_counts` vs `category_counts` and "
    "`multi_fire_patterns`) — these are cases where the precommit's ordering "
    "materially affected the result; (4) the connection to R2-Q1's cross-host "
    "transfer findings; (5) the methodological honesty about working-set filtering "
    "(n=" + str(n_filtered_out) + " images excluded for ill-posed segmentation)."
)

# ----- Assemble and save -----
taxonomy_distribution = {
    "results": results,
    "methodology": methodology,
    "interpretation": interpretation,
}

json_path = OUTPUT_DIR / "taxonomy_distribution.json"
with open(json_path, "w") as f:
    json.dump(taxonomy_distribution, f, indent=2)
print(f"Saved {json_path}\n")

# ----- Plot -----
# Sort the four named categories by count descending, then anchor
# other_ambiguous at the right with a visually distinct color.
named_categories = [name for name, _ in PREDICATE_ORDER]
named_sorted = sorted(named_categories, key=lambda c: category_counts[c], reverse=True)
plot_order = named_sorted + ["other_ambiguous"]
plot_counts = [category_counts[c] for c in plot_order]
plot_labels = [c.replace("_", "\n") for c in plot_order]

# Color scheme: a single accent color for the four named predicates,
# gray for other_ambiguous to mark it as the residual bucket.
NAMED_COLOR = "#3b6fa3"
OTHER_COLOR = "#9a9a9a"
bar_colors = [NAMED_COLOR] * len(named_sorted) + [OTHER_COLOR]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(plot_labels, plot_counts, color=bar_colors)

# Value labels on top of each bar.
for bar, count in zip(bars, plot_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(plot_counts) * 0.01,
        str(count),
        ha="center",
        va="bottom",
        fontsize=11,
    )

ax.set_ylabel("Number of misclassifications", fontsize=11)
ax.set_title(
    f"R2-Q2: Failure-mode distribution across {n_analyzed} PD misclassifications",
    fontsize=12,
)
ax.set_ylim(0, max(plot_counts) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xticks(fontsize=9)
plt.tight_layout()

plot_path = OUTPUT_DIR / "taxonomy_distribution.png"
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

# ----- Final summary print -----
print(f"\n{'─' * 60}")
print(f"R2-Q2 N04 complete. Final deliverables in {OUTPUT_DIR}:")
print(f"{'─' * 60}")
print(f"  categorizations.parquet      ({len(categorizations)} rows, audit trail)")
print(f"  taxonomy_distribution.json   (results / methodology / interpretation)")
print(f"  taxonomy_distribution.png    (five-bar chart for the paper)")
print(f"\nHeadline finding:")
top_category, top_count = max(category_counts.items(), key=lambda kv: kv[1])
print(f"  {top_category}: {top_count}/{n_analyzed} ({100*top_count/n_analyzed:.0f}%)")

## 9 — Where this goes next

N04 is done. The eight preceding sections turned a Week-1 precommit into a paper-ready failure-mode distribution. This section orients you toward Week 5 — the paper-writing work that uses what N04 produced.

### What you have, on disk

Three artifacts in `OUTPUT_DIR` are the basis for Week 5's paper:

- **`taxonomy_distribution.json`** — the reader-facing summary. Results, methodology, and an interpretation placeholder. Your first writing task is filling in that placeholder.
- **`taxonomy_distribution.png`** — the paper's central figure. Five-bar distribution chart.
- **`categorizations.parquet`** — the audit trail. One row per analyzed image; every derived quantity, every predicate firing, every category assignment is in there. The paper won't reproduce this table, but any reviewer question of the form "why was image X classified as Y" gets answered from here.

Plus the upstream evidence: `sam_visual_check.png`, `sam_borderline_check.png`, `working_set_unfiltered.parquet`, `analysis_ready.parquet`, `leaf_masks_metadata.parquet`, `derived_quantities.parquet`, `predicate_results.parquet`. Most of these don't appear in the paper, but they're how you defend the methodology if asked.

### The headline finding, plainly stated

Look at the JSON's `category_counts`. One number dominates the others by an order of magnitude. That number is your paper's central result. Your interpretation needs to say what it means — not just restate it, but explain what it implies about how the model fails.

### The R2-Q1 ↔ R2-Q2 connection

R2-Q1 asked whether a PV-trained classifier transfers to PD, and where the transfer breaks down. R2-Q2 asked *why* it breaks down. The two findings need to talk to each other in your paper's discussion.

A few connecting questions to think through:

- **R2-Q1 found that some categories transfer better than others.** R2-Q2 found that misclassifications are dominated by a particular failure mode. Are those two findings consistent? Does the dominant failure mode help explain which categories failed to transfer?
- **Both findings can be read against the PV shortcut-learning hypothesis.** Noyan (2022) showed PV has an eight-pixel background-feature shortcut. If a PV-trained model has internalized that shortcut, what would you predict R2-Q1 and R2-Q2 to show? How closely does what you observed match that prediction?
- **The paper's argument is stronger if R2-Q1 and R2-Q2 reinforce each other than if they're treated as two independent findings.** Where does each one corroborate the other?

### Caveats your discussion section needs to acknowledge

Three honest disclosures, all sourced from artifacts N04 already produced:

1. **The working set was filtered.** 13 of 81 misclassifications were excluded because SAM produced a degenerate mask (`fg_fraction` outside `[0.05, 0.85]`). Inspection of the dropped images showed they were ill-posed segmentation problems — wide field shots, multi-leaf compositions, low-contrast foliage-on-foliage — rather than random failures. The analysis is therefore restricted to images where the segmentation problem was well-posed. Note this in methods; don't bury it.

2. **The predicate scheme has known dependencies.** Look at `multi_fire_patterns` in the JSON. If one pattern accounts for most multi-fires (e.g., predicate A and predicate B co-firing on N images), those predicates aren't independent — they're picking up correlated signal in different ways. Under the precommit's first-match-wins ordering, all of those went to `other_ambiguous`. A different ordering would have classified them differently. The discussion should name this honestly: results are conditional on the pre-committed predicate order.

3. **The methodology diverges from the precommit in two places.** N01 committed to (a) IoU-based validation of SAM masks against a hand-segmented reference set, and (b) a color-thresholding fallback for images where SAM failed. N04 executed (a) as a qualitative visual inspection and (b) as a filter excluding ill-posed images. Both deviations are defensible — and are documented in Section 4's opener and Section 5's opener — but they're real deviations from a locked plan. The methods section needs to be transparent about this, not silently substitute the new procedure.

### What your Week 5 deliverable looks like

You'll produce two artifacts in Week 5:

- A **paper** synthesizing R2-Q1 and R2-Q2 into a unified argument about how a PV-trained ResNet-50 generalizes (or fails to) on PlantDoc.
- A **presentation** of the same material, ~15 minutes.

Both rely on the artifacts N04 produced. The paper-writing scaffold from the program will tell you the structure; your job is to put real findings into it.

### Before you start writing

Three things to do before you draft a single paragraph:

1. **Read the JSON's `methodology` section out loud.** If a sentence there doesn't make sense to you, you don't understand the methodology well enough to write about it. Go back to the relevant section of N04 and re-read until it clicks.
2. **Pull the `categorizations.parquet` into a notebook and look at five or six random rows.** For each, check that the assigned category matches what the per-image evidence shows. If a row's assignment looks wrong to you, that's a signal to look harder — either at that image or at the predicate logic.
3. **Read R2-Q1's `gap_decomposition.parquet` and `characterization_summary.json` alongside R2-Q2's outputs.** Notice what each says. Notice what they don't say. The connections between them are what your paper is about.

---

*— End of N04 —*